# Chapter 7: GRPO and Reasoning Models
### **RL: The Seminal Papers** by Rahul Shirale

This notebook trains a language model with **Group Relative Policy Optimization**
to emit JSON that satisfies a strict schema, using nothing but a rule-based reward.

It is deliberately a miniature of DeepSeek-R1-Zero: we start from a *base* model that
has never been instruction-tuned, and the only training signal is a Python function
that scores the output. No demonstrations, no preference labels, no critic network.

**What you will see.** The base model produces fully schema-compliant JSON a little over
40% of the time. Everything else is close but wrong in ways a parser rejects: `TicketId` instead
of `ticket_id`, `Critical` instead of `critical`, the object wrapped in an array, or a
trailing sentence after the closing brace. GRPO's job is to move that 40% up, and the
reward function is the only thing telling it what 'up' means. Forty steps took it from
42% to 83% on the run recorded in this notebook.


## 1. Why these three libraries

Each dependency maps onto one part of the algorithm, which is worth being explicit about
before we install anything:

| library | what it contributes |
|---|---|
| **torch** | Tensors and autograd. The GRPO objective itself &mdash; group standardization, the clipped ratio, the KL penalty &mdash; is plain PyTorch and nothing else. |
| **transformers** | The model and tokenizer. `generate()` samples the *G* candidates per prompt; a forward pass gives the per-token log-probabilities the objective needs. |
| **peft** | LoRA. Freezes the base weights and trains a thin set of adapters, so the run fits on one GPU. |

`peft` earns its place twice over, and the second reason is specific to GRPO. The KL term
in the objective needs a frozen reference policy. Without LoRA you load a second copy of
the weights to get it. With LoRA the base weights *are* the reference: call
`disable_adapter()` and the same model answers as the frozen policy. One context manager
replaces an entire second model in memory.


In [ ]:
# In Google Colab, torch is pre-installed; the other two are not.
# The specifiers must stay quoted -- an unquoted >= is read by the shell as a
# redirection. These bounds match requirements-llm.txt in the repository.
# !pip install -q "transformers>=4.44,<5.0" "peft>=0.12,<1.0"

import gc, json, os, random, time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Part of seeding, not a performance tweak. Torch's intra-op parallelism changes
# the order floating-point work is reduced in, so the same seed on a different
# core count produces different logits and therefore different sampled tokens.
# This notebook is standalone by design and imports nothing from src/, so it
# duplicates the line that src/part_2_methods/ch07_grpo/seeding.py holds for the
# modules.
torch.set_num_threads(1)

print('torch', torch.__version__)


## 2. Choosing a device

One measured warning. On an Apple Silicon Mac, MPS is **not** automatically the fast
choice for a model this small: benchmarked on a 0.5B model in float32, MPS produced
4.2 tokens/second against 7.4 on plain CPU, because several generation ops fall back to
the CPU anyway. MPS also needs `PYTORCH_ENABLE_MPS_FALLBACK=1` and an explicit
`attention_mask` on torch below 2.4, or generation raises.

So: CUDA if you have it, otherwise CPU. Set `FORCE_MPS = True` only if you want to
experiment.


In [ ]:
FORCE_MPS = bool(os.environ.get('CH7_FORCE_MPS'))

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif FORCE_MPS and torch.backends.mps.is_available():
    os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

# A base model, NOT the -Instruct variant. Instruction tuning already solves this task
# (measured: 20/20 perfect), which would leave GRPO nothing to learn.
# The environment variable exists so the test suite can point this at a tiny
# randomly-initialized model; leave it unset to run the chapter.
MODEL_ID = os.environ.get('CH7_MODEL_ID', 'Qwen/Qwen2.5-0.5B')

print('device:', DEVICE)
print('model :', MODEL_ID)


## 3. The task and the data

We generate the dataset locally rather than downloading one. It is reproducible from the
seed, it cannot rot when a dataset is renamed, and it keeps the notebook self-contained.

Each example is an unstructured support-ticket note plus the JSON we want back. The
schema is small but strict: four keys exactly, a severity drawn from a fixed set, and a
real boolean rather than the string `"true"`.


In [ ]:
SCHEMA_KEYS = ['ticket_id', 'severity', 'component', 'resolved']
SEVERITIES  = ['low', 'medium', 'high', 'critical']
COMPONENTS  = ['auth service', 'billing page', 'search index', 'checkout flow',
               'email worker', 'admin console', 'image uploader', 'report builder']

TEMPLATES = [
    'Ticket {tid} came in about the {comp} failing. Marked {sev}. {state}.',
    'Issue {tid}: users report the {comp} misbehaving. Priority {sev}. {state}.',
    'Report {tid} - the {comp} is degraded, {sev} severity, {state}.',
    'Case {tid}: {comp} returned errors for several customers. Severity {sev}. {state}.',
]

def make_dataset(n=64, seed=SEED):
    rng = random.Random(seed)
    rows = []
    for _ in range(n):
        tid  = str(rng.randint(1000, 9999))
        sev  = rng.choice(SEVERITIES)
        comp = rng.choice(COMPONENTS)
        done = rng.random() < 0.5
        text = rng.choice(TEMPLATES).format(
            tid=tid, comp=comp, sev=sev,
            state='Resolved' if done else 'Still open')
        rows.append({
            'text': text,
            'target': {'ticket_id': tid, 'severity': sev,
                       'component': comp, 'resolved': done},
        })
    return rows

DATA = make_dataset()
print(f'{len(DATA)} examples')
print(DATA[0]['text'])
print(DATA[0]['target'])


## 4. The reward function

This is the whole training signal. It is the same composite reward developed in the
chapter: structural validity first, then schema conformance, then content accuracy, with
graduated penalties rather than all-or-nothing zeros so that a nearly-correct answer
still carries gradient.

Note the deliberate `-1.0` for output that does not start with `{`. The base model likes
to preamble, and without this term it learns it can chat its way to a passing score.


In [ ]:
def extract_json(text):
    '''Return the first balanced {...} block, or None.'''
    start = text.find('{')
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


def compute_json_reward(output_text, target):
    '''Score one completion. Higher is better; range is roughly -3 to +6.'''
    reward = 0.0
    text = output_text.strip()

    # Format: raw JSON only, no prose and no markdown fence.
    if text.startswith('{'):
        reward += 1.0
    else:
        reward -= 1.0

    blob = extract_json(text)
    if blob is None:
        return reward - 2.0
    try:
        data = json.loads(blob)
    except json.JSONDecodeError:
        return reward - 2.0
    if not isinstance(data, dict):
        return reward - 1.0
    reward += 1.0                                   # parsed as an object

    missing = set(SCHEMA_KEYS) - set(data)
    extra   = set(data) - set(SCHEMA_KEYS)
    reward += 1.0 if not missing else -0.5 * len(missing)
    reward -= 0.2 * len(extra)

    if str(data.get('severity', '')) in SEVERITIES:  # exact case required
        reward += 0.5
    if isinstance(data.get('resolved'), bool):
        reward += 0.5

    for key in SCHEMA_KEYS:                          # content accuracy
        if key in data and data[key] == target[key]:
            reward += 0.5
    return reward


def is_compliant(output_text, target):
    '''Strict pass/fail used for reporting, not for training.'''
    blob = extract_json(output_text.strip())
    if blob is None or not output_text.strip().startswith('{'):
        return False
    try:
        d = json.loads(blob)
    except json.JSONDecodeError:
        return False
    return (isinstance(d, dict) and set(d) == set(SCHEMA_KEYS)
            and str(d.get('severity')) in SEVERITIES
            and isinstance(d.get('resolved'), bool))


# Sanity checks on hand-written cases before we trust it on model output.
good = json.dumps(DATA[0]['target'])
print('perfect      ', round(compute_json_reward(good, DATA[0]['target']), 2))
print('fenced       ', round(compute_json_reward('```json\n' + good, DATA[0]['target']), 2))
print('wrong keys   ', round(compute_json_reward('{"TicketId": "1"}', DATA[0]['target']), 2))
print('not json     ', round(compute_json_reward('Sure! Here you go.', DATA[0]['target']), 2))


## 5. Loading the model with LoRA

`get_peft_model` freezes every base weight and inserts trainable rank-16 adapters into
the attention projections. The printout below is the memory argument from the chapter made
concrete: we train a fraction of a percent of the parameters.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'left'          # required for batched generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(base, lora_cfg).to(DEVICE)
model.print_trainable_parameters()

PROMPT = ('Convert the record to JSON with keys ticket_id, severity, component, '
          'resolved.\nRecord: {r}\nJSON: ')


## 6. Sampling a group

GRPO needs *G* completions for the same prompt so it can compare them against each other.
We generate them as one batch, which is what makes the extra samples affordable: the cost
is far less than *G* separate calls.


In [ ]:
MAX_NEW_TOKENS = int(os.environ.get('CH7_MAX_NEW_TOKENS', 72))


@torch.no_grad()
def sample_group(prompt_text, n, max_new_tokens=MAX_NEW_TOKENS, temperature=0.9):
    '''Return n sampled completions for one prompt.'''
    enc = tokenizer([prompt_text] * n, return_tensors='pt', padding=True).to(DEVICE)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=True,
        temperature=temperature, top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[:, enc['input_ids'].shape[1]:]
    texts = tokenizer.batch_decode(gen, skip_special_tokens=True)
    return texts, enc['input_ids'], gen


def evaluate(rows, n_each=4):
    '''Fraction of completions that are strictly schema-compliant.'''
    hits = total = 0
    for row in rows:
        texts, _, _ = sample_group(PROMPT.format(r=row['text']), n_each)
        for t in texts:
            hits += is_compliant(t, row['target'])
            total += 1
    return hits / total


## 7. Baseline

Measure before training, so the number after training means something. Expect roughly
0.4 &mdash; the model can nearly do this, and nearly is what the reward will attack.


In [ ]:
EVAL_ROWS = DATA[:int(os.environ.get('CH7_EVAL_ROWS', 6))]

t0 = time.time()
baseline = evaluate(EVAL_ROWS)
print(f'baseline compliance: {baseline:.0%}   ({time.time() - t0:.0f}s)')

# Look at what it actually produces, because the failure modes are the point.
texts, _, _ = sample_group(PROMPT.format(r=EVAL_ROWS[0]['text']), 4)
for t in texts:
    print(' ', repr(t[:96]))


## 8. Log-probabilities, and the free reference policy

The objective needs the log-probability of each generated token under two policies: the
one we are training, and a frozen reference. `disable_adapter()` gives us the second for
nothing &mdash; inside that context the adapters are bypassed and the model *is* the
original base weights.

Note the **mean** on the last line, not a sum. Averaging within a sample means each
token of a long completion contributes less to the loss than each token of a short one,
which teaches the model to answer in as few tokens as possible. That is right for a
one-line JSON object and wrong for long chain-of-thought &mdash; it is GRPO's documented
short-response bias, and exactly what DAPO's token-level aggregation corrects. Sum
instead of mean if your task rewards length.


In [ ]:
def sequence_logprob(prompt_ids, gen_ids, use_adapter=True):
    '''Mean per-token log-prob of gen_ids given prompt_ids, per sequence.'''
    full = torch.cat([prompt_ids, gen_ids], dim=1)
    ctx = torch.enable_grad() if use_adapter else torch.no_grad()
    with ctx:                                      # <-- the gather too, not just the forward
        if use_adapter:
            logits = model(full).logits
        else:
            with model.disable_adapter():          # <-- the frozen reference policy
                logits = model(full).logits

        logits = logits[:, prompt_ids.shape[1] - 1:-1, :]
        logp = torch.log_softmax(logits.float(), dim=-1)
        token_logp = logp.gather(-1, gen_ids.unsqueeze(-1)).squeeze(-1)

        mask = (gen_ids != tokenizer.pad_token_id).float()
        return (token_logp * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)


## 9. The GRPO objective

This is listing 7.4 from the chapter, unchanged. Three things are worth re-reading here:

- The advantage is standardized **within the group**. That pair of statistics is the entire
  replacement for a critic network.
- The `1e-4` in the denominator guards the case where every candidate scores identically.
  It happens once a model gets good at easy prompts, and it is the degenerate case DAPO's
  dynamic sampling was designed to remove.
- The KL term uses Schulman's unbiased estimator, `exp(u) - u - 1`, which is never
  negative. Averaging the raw log-ratio instead would let the penalty go negative and
  *reward* the policy for drifting away from the reference.


In [ ]:
def grpo_loss(logp, old_logp, ref_logp, rewards, group_size,
              clip_eps=0.2, kl_beta=0.04):
    K = logp.shape[0] // group_size
    r_g = rewards.view(K, group_size)

    mean = r_g.mean(dim=1, keepdim=True)
    std = r_g.std(dim=1, keepdim=True)
    adv = ((r_g - mean) / (std + 1e-4)).view(-1)

    ratio = torch.exp(logp - old_logp)
    clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)
    loss_pg = -torch.min(ratio * adv, clipped * adv).mean()

    log_r = ref_logp - logp
    kl = torch.exp(log_r) - log_r - 1.0

    return loss_pg + kl_beta * kl.mean()


## 10. The training loop

Now the pieces assemble into the loop from listing 7.3. Per step: sample a group, score it,
take log-probs under both policies, and update.

Because we make a single update per exploration stage &mdash; the setting the DeepSeek team
used &mdash; the behavior policy and the current policy are identical when the ratio is
computed, so `old_logp` is just `logp` detached and the clip never binds. That is not a
shortcut; it is what the objective reduces to at &mu; = 1.

**Watch the gradient norm, not the loss.** Because we take a single update per exploration
stage, `old_logp` equals `logp` at the moment the ratio is formed, so the ratio is exactly 1
and the surrogate reduces to `-mean(adv)`. The advantages are standardized within the group,
so that mean is *identically zero* &mdash; the loss recorded in `history` is 0.0000 on every
step and that is correct, not broken. The gradient is not zero: differentiating through
`exp(logp - old_logp)` gives `-adv/N` per sample, which is exactly the REINFORCE-with-baseline
update. So we report the gradient norm, which actually moves.

**Runtime.** Measured on this machine (CPU, 0.5B): the first step costs about 40 seconds
including warmup, after which steps settle to roughly 25 seconds. The 40 steps below took
16 minutes on CPU; a CUDA GPU is several times faster.


In [ ]:
STEPS      = int(os.environ.get('CH7_STEPS', 40))
GROUP_SIZE = int(os.environ.get('CH7_GROUP_SIZE', 8))      # G
LR         = 1e-5

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR)

history = []
t_start = time.time()

for step in range(STEPS):
    row = DATA[step % len(DATA)]
    prompt = PROMPT.format(r=row['text'])

    # 1. Sample a group of candidates from the current policy.
    texts, prompt_ids, gen_ids = sample_group(prompt, GROUP_SIZE)

    # 2. Score every candidate with the rule-based reward.
    rewards = torch.tensor(
        [compute_json_reward(t, row['target']) for t in texts],
        dtype=torch.float32, device=DEVICE)

    # A group where everything scores the same carries no signal: the advantage
    # is zero for every member. Skip it rather than take a null step.
    if rewards.std() < 1e-6:
        history.append({'step': step, 'reward': rewards.mean().item(),
                        'loss': None, 'skipped': True})
        continue

    # 3. Log-probs under the current policy and the frozen reference.
    logp = sequence_logprob(prompt_ids, gen_ids, use_adapter=True)
    ref_logp = sequence_logprob(prompt_ids, gen_ids, use_adapter=False)
    old_logp = logp.detach()

    # 4. GRPO objective and update.
    loss = grpo_loss(logp, old_logp, ref_logp, rewards, GROUP_SIZE)
    optimizer.zero_grad()
    loss.backward()
    gnorm = torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 1.0)
    optimizer.step()

    history.append({'step': step, 'reward': rewards.mean().item(),
                    'loss': loss.item(), 'gnorm': gnorm.item(),
                    'skipped': False})

    if step % 5 == 0 or step == STEPS - 1:
        el = time.time() - t_start
        print(f'step {step:3d}  reward {rewards.mean():6.2f}  '
              f'grad {gnorm:6.3f}  ({el:.0f}s)')

print(f'\ndone in {time.time() - t_start:.0f}s')


## 11. Did it work?

Re-measure with the same evaluation used for the baseline. The move can be large &mdash; this run went from 42% to
83% in 40 steps &mdash; because the base model was already close and the reward only had to
teach it which of its own outputs to prefer. Two honest caveats before you read the
number: RL at this scale is noisy, so run it twice with different seeds before believing
any single figure, and a jump this size says the task was nearly solved already, not that
40 steps of GRPO can teach a 0.5B model something it could not do.


In [ ]:
after = evaluate(EVAL_ROWS)
print(f'baseline compliance: {baseline:.0%}')
print(f'after {STEPS} steps    : {after:.0%}')
print(f'change             : {after - baseline:+.0%}')

rewarded = [h['reward'] for h in history if not h['skipped']]
if rewarded:
    half = max(1, len(rewarded) // 2)
    print(f'\nmean reward, first half: {sum(rewarded[:half]) / half:.2f}')
    print(f'mean reward, last half : {sum(rewarded[half:]) / len(rewarded[half:]):.2f}')
print(f'steps skipped (zero variance): '
      f"{sum(h['skipped'] for h in history)}/{len(history)}")


In [ ]:
# %matplotlib inline
# Optional: plot the reward curve. The magic is commented out rather than
# deleted so this cell runs unchanged in-process under pytest; uncomment it if
# your Jupyter front end needs it.
try:
    import matplotlib.pyplot as plt
    xs = [h['step'] for h in history if not h['skipped']]
    ys = [h['reward'] for h in history if not h['skipped']]
    plt.figure(figsize=(7, 3))
    plt.plot(xs, ys, marker='o', linewidth=1)
    plt.xlabel('step'); plt.ylabel('mean group reward')
    plt.title('GRPO on strict JSON schema adherence')
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed; skipping plot')


## 12. Freeing memory

Notebooks keep every object alive until the kernel restarts, so re-running the model-loading
cell without clearing the old one is the fastest way to exhaust a GPU. Run this before
loading anything else.


In [ ]:
del model, base, optimizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('released')


## What to take away

The algorithm in section 9 is about fifteen lines, and there is no critic anywhere in this
notebook. The baseline was replaced by two numbers &mdash; the mean and standard deviation
of a group of samples &mdash; and that substitution is the whole of GRPO.

Everything else here is scaffolding you would swap out for a real task: the dataset, the
reward function, the model. The reward is the part worth your attention, because in GRPO it
is the *only* thing that defines correct. Ours is about forty lines and it still needed a
`-1.0` term to stop the model preambling its way to a passing score. That is the lesson
the chapter's production case study makes at scale: when the output is machine-checkable,
a small model trained against a precise rule can beat a much larger general one &mdash; but
only to the extent the rule is actually precise.
